In [2]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM  # Punkt 4 benötigt
import pickle

# Load from a file
with open('../Dataset/1K_sample.pkl', 'rb') as file:
    df = pickle.load(file)

# ============================================
# 1. Chunking
# ============================================
chunks = []
metadaten = []  # parallel zu chunks speichern

for idx, row in df.iterrows():
    abstract = row['abstract']
    # Einfaches Chunking nach Sätzen
    for satz in abstract.split('. '):
        if satz:
            chunks.append(satz.strip())
            # Metadaten zu diesem Chunk speichern
            metadaten.append({
                'pmcid': row['pmcid'],
                'title': row['title'],
                'year': row['year'],
                'chunk_index': len([c for c in chunks if c.endswith(satz)])  # vereinfacht
            })

# ============================================
# 2. Embedding
# ============================================
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embed_model.encode(chunks)

# ============================================
# 3. Retrieval mit Metadaten
# ============================================
def retrieve_with_metadata(query, k=2):
    query_emb = embed_model.encode([query])[0]
    similarities = np.dot(chunk_embeddings, query_emb) / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_emb)
    )
    top_k_idx = np.argsort(similarities)[-k:][::-1]
    
    # Chunks UND Metadaten zurückgeben
    result_chunks = [chunks[i] for i in top_k_idx]
    result_metadata = [metadaten[i] for i in top_k_idx]
    return result_chunks, result_metadata

# ============================================
# 4. LLM laden (hier FLAN-T5-small)
# ============================================
print("Lade LLM...")
llm_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(llm_name)

# ============================================
# 5. Antwortgenerierung definieren
# ============================================
def generate_answer(query):
    # Relevante Chunks holen (Korrektur: Funktion heißt retrieve_with_metadata)
    gefundene, _ = retrieve_with_metadata(query, k=2)
    kontext = " ".join(gefundene)

    # Prompt im Stil von FLAN-T5 (instruction-tuned)
    prompt = f"Answer the question based on the given context.\nContext: {kontext}\nQuestion: {query}\nAnswer:"

    # LLM aufrufen
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = llm.generate(
        **inputs, 
        max_new_tokens=100, 
        do_sample=True,        # WICHTIG: erst damit wird Temperatur aktiv
        temperature=0.3
    )
    antwort = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return antwort



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10091.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Lade LLM...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 8454.28it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
